# English to Japanese Translations Chunking

In [1]:
from pathlib import Path
import pandas as pd

# Resolve dataset path robustly whether the notebook runs from project root or rag/.
candidate_paths = [
    Path("kb/eng-jap.tsv"),
    Path("../kb/eng-jap.tsv"),
]

dataset_path = next((p for p in candidate_paths if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError(
        "Could not find eng-jap.tsv. Checked: kb/eng-jap.tsv and ../kb/eng-jap.tsv"
    )

print(f"Using dataset: {dataset_path.resolve()}")

# TSV format: English \t Japanese
df = pd.read_csv(
    dataset_path,
    sep="\t",
    header=None,
    names=["eng_id","english","jp_id","japanese"],
    dtype=str,
).fillna("")

df.head(5)

Using dataset: D:\GitHub\is469\kb\eng-jap.tsv


,eng_id,english,jp_id,japanese
0,1276,Let's try something.,4702,何かしてみましょう。
1,1277,I have to go to sleep.,4703,私は眠らなければなりません。
2,1277,I have to go to sleep.,344904,そろそろ寝なくちゃ。
3,1280,Today is June 18th and it is Muiriel's birthday!,4705,今日は６月１８日で、ムーリエルの誕生日です！
4,1282,Muiriel is 20 now.,4707,ムーリエルは２０歳になりました。


In [13]:
# Dataset overview
print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print()

# Missing/empty checks
empty_english = (df["english"].str.strip() == "").sum()
empty_japanese = (df["japanese"].str.strip() == "").sum()
print("Empty text counts:")
print(f"- english:  {empty_english:,}")
print(f"- japanese: {empty_japanese:,}")
print()

# Duplicate pair checks
duplicate_pairs = df.duplicated(subset=["english", "japanese"]).sum()
print(f"Duplicate EN-JP pairs: {duplicate_pairs:,}")
print()

# Length diagnostics (characters)
length_stats = df.assign(
    english_chars=df["english"].str.len(),
    japanese_chars=df["japanese"].str.len(),
)[["english_chars", "japanese_chars"]].describe(percentiles=[0.5, 0.9, 0.99])
print("Character length summary:")
display(length_stats)

# Quick sample
print("Sample rows:")
display(df.sample(5, random_state=42))

Rows: 279,694
Columns: ['eng_id', 'english', 'jp_id', 'japanese']

Empty text counts:
- english:  0
- japanese: 0

Duplicate EN-JP pairs: 2

Character length summary:


,english_chars,japanese_chars
count,279694.000000,279694.000000
mean,37.889168,17.210712
std,32.721087,7.997025
min,3.000000,2.000000
50%,34.000000,16.000000
90%,57.000000,26.000000
99%,107.000000,44.000000
max,13653.000000,302.000000


Sample rows:


,eng_id,english,jp_id,japanese
27874,41423,Don't keep company with such a bad boy.,204181,そんな悪い子と友達になるな。
216883,9354007,It wasn't our fault. Believe me.,9678940,私達のせいじゃないわ。信じてよ。
57687,71054,May I ask you a question?,149509,質問をしてもいいですか。
86982,264505,"In his autobiography, he repeatedly refers to ...",150052,自伝の中で彼はくりかえし不幸な少年時代に言及している。
254190,9193501,The apple cake's ran out.,11563342,アップルケーキがなくなっちゃった。


In [14]:
df.head(5)

,eng_id,english,jp_id,japanese
0,1276,Let's try something.,4702,何かしてみましょう。
1,1277,I have to go to sleep.,4703,私は眠らなければなりません。
2,1277,I have to go to sleep.,344904,そろそろ寝なくちゃ。
3,1280,Today is June 18th and it is Muiriel's birthday!,4705,今日は６月１８日で、ムーリエルの誕生日です！
4,1282,Muiriel is 20 now.,4707,ムーリエルは２０歳になりました。


In [2]:
import re
from typing import Any


def _clean_text(s: str) -> str:
    s = s if isinstance(s, str) else ""
    s = re.sub(r"\s+", " ", s).strip()
    return s


def chunk_parallel_df(
    df: pd.DataFrame,
    *,
    chunk_size: int = 8,
    overlap: int = 2,
    min_pairs: int = 3,
) -> pd.DataFrame:
    """Chunk EN-JA sentence pairs into retrieval-ready documents.

    Each chunk keeps multiple parallel pairs so rerankers can score richer context.
    """
    if chunk_size <= 0:
        raise ValueError("chunk_size must be > 0")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and < chunk_size")

    # Keep only rows with both sides present.
    clean_df = df.copy()
    clean_df["english"] = clean_df["english"].map(_clean_text)
    clean_df["japanese"] = clean_df["japanese"].map(_clean_text)
    clean_df = clean_df[(clean_df["english"] != "") & (clean_df["japanese"] != "")].reset_index(drop=True)

    step = chunk_size - overlap
    chunk_rows: list[dict[str, Any]] = []

    for start in range(0, len(clean_df), step):
        end = min(start + chunk_size, len(clean_df))
        window = clean_df.iloc[start:end]
        if len(window) < min_pairs:
            continue

        lines = []
        pair_items: list[dict[str, str]] = []
        for i, row in enumerate(window.itertuples(index=False), start=1):
            en = row.english
            ja = row.japanese
            lines.append(f"[{i}] EN: {en}\n[{i}] JA: {ja}")
            pair_items.append({"english": en, "japanese": ja})

        chunk_id = f"engjap_chunk_{start:06d}_{end - 1:06d}"
        chunk_text = "\n\n".join(lines)

        chunk_rows.append(
            {
                "chunk_id": chunk_id,
                "start_row": int(start),
                "end_row": int(end - 1),
                "n_pairs": int(len(window)),
                "chunk_text": chunk_text,
                "pairs": pair_items,
            }
        )

    return pd.DataFrame(chunk_rows)


# Tuned for your 2-stage retrieval pipeline.
BI_ENCODER_K = 20
RERANK_TOP_N = 3

chunks_df = chunk_parallel_df(df, chunk_size=8, overlap=2, min_pairs=3)
print(f"Built {len(chunks_df):,} chunks from {len(df):,} rows")
print(f"Bi-encoder retrieve: top {BI_ENCODER_K}; cross-encoder keep: top {RERANK_TOP_N}")

display(chunks_df[["chunk_id", "start_row", "end_row", "n_pairs"]].head())
print("\nSample chunk text:\n")
print(chunks_df.loc[0, "chunk_text"][:1200] if len(chunks_df) else "No chunks generated.")

Built 46,616 chunks from 279,694 rows
Bi-encoder retrieve: top 20; cross-encoder keep: top 3


,chunk_id,start_row,end_row,n_pairs
0,engjap_chunk_000000_000007,0,7,8
1,engjap_chunk_000006_000013,6,13,8
2,engjap_chunk_000012_000019,12,19,8
3,engjap_chunk_000018_000025,18,25,8
4,engjap_chunk_000024_000031,24,31,8



Sample chunk text:

[1] EN: Let's try something.
[1] JA: 何かしてみましょう。

[2] EN: I have to go to sleep.
[2] JA: 私は眠らなければなりません。

[3] EN: I have to go to sleep.
[3] JA: そろそろ寝なくちゃ。

[4] EN: Today is June 18th and it is Muiriel's birthday!
[4] JA: 今日は６月１８日で、ムーリエルの誕生日です！

[5] EN: Muiriel is 20 now.
[5] JA: ムーリエルは２０歳になりました。

[6] EN: Muiriel is 20 now.
[6] JA: 今、ムーリイエルは２０歳だ。

[7] EN: The password is "Muiriel".
[7] JA: パスワードは「Muiriel」です。

[8] EN: I will be back soon.
[8] JA: すぐに戻ります。


In [ ]:
ids = chunks_df["chunk_id"].tolist()
documents = chunks_df["chunk_text"].tolist()
metadatas = chunks_df[["chunk_id", "start_row", "end_row", "n_pairs"]].to_dict("records")

print(f"Prepared {len(documents):,} documents for indexing")
print("Example metadata:", metadatas[0] if metadatas else {})

Prepared 46,616 documents for indexing
Example metadata: {'chunk_id': 'engjap_chunk_000000_000007', 'start_row': 0, 'end_row': 7, 'n_pairs': 8}


In [4]:
import json
from pathlib import Path

# Export chunked EN-JA retrieval docs to JSONL
chunk_out_path = Path("../kb/eng_jap_chunks.jsonl")
chunk_out_path.parent.mkdir(parents=True, exist_ok=True)

records = chunks_df[["chunk_id", "start_row", "end_row", "n_pairs", "chunk_text", "pairs"]].to_dict("records")

with chunk_out_path.open("w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Saved {len(records):,} chunk records -> {chunk_out_path.resolve()}")

Saved 46,616 chunk records -> D:\GitHub\is469\kb\eng_jap_chunks.jsonl


## PDF Chunking for RAG Knowledge Base

This section processes two PDF documents into chunked `.jsonl` files for a RAG knowledge base:

1. **JTF Style Guide** (`jtf_style_guide_e.pdf`) — chunked by numbered section headings
2. **Tae Kim's Grammar Guide** (`grammar_guide.pdf`) — selected sections only: particle usage (は/も/が), polite vs plain form, transitive/intransitive verbs

Each chunk is a dict with fields: `source`, `section`, `text`, `chunk_id`. Outputs are saved to `../kb/`.

In [15]:
%pip install -q pdfplumber

import pdfplumber
import json
import re
from pathlib import Path

KB_DIR = Path("../kb")
KB_DIR.mkdir(parents=True, exist_ok=True)


def extract_pdf_text(pdf_path):
    """Extract text from every page of a PDF, returned as one string."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                pages.append(text)
    return "\n".join(pages)


def clean_pdf_text(text):
    """Clean common PDF extraction artifacts.

    Full-width numbers/letters are preserved — in these documents
    they are intentional examples, not extraction errors.
    """
    text = re.sub(r"--\s*\d+\s+of\s+\d+\s*--", "", text)
    text = text.replace("\ufb01", "fi").replace("\ufb02", "fl")
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2013", "-").replace("\u2014", "--")
    text = text.replace("\x0c", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"\n\d{1,3}\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def split_sentences(text):
    """Split mixed EN/JP text into sentence-level units."""
    paragraphs = re.split(r"\n\s*\n", text.strip())
    sentences = []
    for para in paragraphs:
        para = para.strip()
        if not para:
            continue
        sents = re.split(
            r"(?<=[。！？])\s*(?=\S)|(?<=[.!?])\s+(?=[A-Z\"'(])",
            para,
        )
        sentences.extend(s.strip() for s in sents if s and s.strip())
    return sentences


def chunk_with_overlap(sentences, max_sents=5, overlap=2):
    """Group sentences into overlapping chunks."""
    if not sentences:
        return []
    if len(sentences) <= max_sents:
        return [" ".join(sentences)]
    chunks, step = [], max(1, max_sents - overlap)
    for i in range(0, len(sentences), step):
        chunk_sents = sentences[i : i + max_sents]
        chunks.append(" ".join(chunk_sents))
        if i + max_sents >= len(sentences):
            break
    return chunks


def estimate_tokens(text):
    """Rough token count (~4 chars / token for mixed EN/JP)."""
    return max(1, len(text) // 4)


def save_jsonl(records, path):
    """Write list of dicts as a .jsonl file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved {len(records)} chunks → {path}")


def print_summary(chunks, label):
    """Print chunk count, average token length, and sample chunks."""
    toks = [estimate_tokens(c["text"]) for c in chunks]
    avg = sum(toks) / len(toks) if toks else 0
    print(f"\n{'=' * 60}")
    print(f"  {label}")
    print(f"{'=' * 60}")
    print(f"  Total chunks : {len(chunks)}")
    print(f"  Avg length   : {avg:.0f} tokens (est.)\n")
    for c in chunks[:3]:
        preview = c["text"][:300] + ("…" if len(c["text"]) > 300 else "")
        print(f"  [{c['chunk_id']}]  section: {c['section']}")
        print(f"  {preview}\n")

Note: you may need to restart the kernel to use updated packages.


### 1. JTF Style Guide (`jtf_style_guide_e.pdf`)

Extract text with `pdfplumber`, split by numbered section headings (e.g. "1.1", "2.1.6"), and create sentence-level chunks with 1–2 sentence overlap. Each section becomes one or more chunks depending on its length.

In [17]:
raw_style = extract_pdf_text("../kb/jtf_style_guide_e.pdf")
style_text = clean_pdf_text(raw_style)

# Match section headings like "1.1. Style", "2.1.6. Chōon ...", etc.
# Requires at least one dot (e.g. "1.1") to avoid matching numbered list items.
heading_re = re.compile(r"^(\d+(?:\.\d+)+)\.?\s+(.+?)$", re.MULTILINE)
matches = list(heading_re.finditer(style_text))
print(f"Found {len(matches)} section headings in JTF style guide\n")

style_chunks = []
source_style = "jtf_style_guide_e.pdf"

for idx, m in enumerate(matches):
    sec_num = m.group(1)
    sec_title = m.group(2).strip()
    heading = f"{sec_num} {sec_title}"

    start = m.start()
    end = matches[idx + 1].start() if idx + 1 < len(matches) else len(style_text)
    body = style_text[start:end].strip()

    sents = split_sentences(body)
    sub_chunks = chunk_with_overlap(sents, max_sents=5, overlap=2)

    for j, ct in enumerate(sub_chunks):
        style_chunks.append({
            "source": source_style,
            "section": heading,
            "text": ct,
            "chunk_id": f"jtf_{sec_num.replace('.', '_')}_{j:02d}",
        })

save_jsonl(style_chunks, KB_DIR / "style_guide_chunks.jsonl")
print_summary(style_chunks, "JTF Style Guide — Chunking Summary")

Found 143 section headings in JTF style guide

Saved 181 chunks → ..\kb\style_guide_chunks.jsonl

  JTF Style Guide — Chunking Summary
  Total chunks : 181
  Avg length   : 63 tokens (est.)

  [jtf_12_5_00]  section: 12.5 12．5
  12.5 12．5
single-byte comma (,)
12. Be consistent in unit nomenclature. Acceptable Unacceptable See
2kg 354m 20℃ 3ft 23坪 14km 5. Writing Units
Is consistently metric. Mixes American, Japanese, and
metric units.

  [jtf_12_5_01]  section: 12.5 12．5
  Writing Units
Is consistently metric. Mixes American, Japanese, and
metric units. Table of Contents
Introduction
1. Sentences ···················································································· 9

  [jtf_1_1_00]  section: 1.1 Style ................................................................................................... 9
  1.1. Style ................................................................................................... 9



### 2. Tae Kim's Grammar Guide (`grammar_guide.pdf`) — Selected Sections

Extract only three targeted topics from the 353-page guide:

- **Particle usage (は/も/が)** — sections 3.3 through 3.3.4
- **Polite vs plain form (です/ます vs だ/である)** — sections 4.1 through 4.1.5
- **Transitive & intransitive verbs** — sections 3.9 through 3.9.1

Same chunking strategy: sentence-level splits with 1–2 sentence overlap.

In [18]:
TARGET_SECTIONS = {
    "3.3", "3.3.1", "3.3.2", "3.3.3", "3.3.4",      # Particles は/も/が
    "3.9", "3.9.1",                                     # Transitive / intransitive
    "4.1", "4.1.1", "4.1.2", "4.1.3", "4.1.4", "4.1.5", # Polite form
}

raw_gram = extract_pdf_text("../kb/grammar_guide.pdf")
gram_text = clean_pdf_text(raw_gram)

# Remove repeating page headers like "3.3. INTRO TO PARTICLES CHAPTER 3. BASIC GRAMMAR"
# These contain section numbers that would confuse heading detection.
gram_text = re.sub(r"^.*CHAPTER \d+\..*$", "", gram_text, flags=re.MULTILINE)

heading_re = re.compile(r"^(\d+(?:\.\d+)+)\.?\s+(.+?)$", re.MULTILINE)
all_matches = list(heading_re.finditer(gram_text))
print(f"Found {len(all_matches)} total section headings in grammar guide")

grammar_chunks = []
source_gram = "grammar_guide.pdf"
kept = 0

for idx, m in enumerate(all_matches):
    sec_num = m.group(1)
    if sec_num not in TARGET_SECTIONS:
        continue
    kept += 1
    sec_title = m.group(2).strip()
    heading = f"{sec_num} {sec_title}"

    start = m.start()
    end = all_matches[idx + 1].start() if idx + 1 < len(all_matches) else len(gram_text)
    body = gram_text[start:end].strip()

    sents = split_sentences(body)
    sub_chunks = chunk_with_overlap(sents, max_sents=5, overlap=2)

    for j, ct in enumerate(sub_chunks):
        grammar_chunks.append({
            "source": source_gram,
            "section": heading,
            "text": ct,
            "chunk_id": f"gram_{sec_num.replace('.', '_')}_{j:02d}",
        })

print(f"Kept {kept} sections matching target topics\n")
save_jsonl(grammar_chunks, KB_DIR / "grammar_chunks.jsonl")
print_summary(grammar_chunks, "Grammar Guide (Selected Sections) — Chunking Summary")

Found 570 total section headings in grammar guide
Kept 26 sections matching target topics

Saved 81 chunks → ..\kb\grammar_chunks.jsonl

  Grammar Guide (Selected Sections) — Chunking Summary
  Total chunks : 81
  Avg length   : 97 tokens (est.)

  [gram_3_3_00]  section: 3.3 Introduction to Particles . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 33
  3.3 Introduction to Particles . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 33

  [gram_3_3_1_00]  section: 3.3.1 Defining grammatical functions with particles . . . . . . . . . . . . . . . . 33
  3.3.1 Defining grammatical functions with particles . . . . . . . . . . . . . . . . 33

  [gram_3_3_2_00]  section: 3.3.2 The 「は」 topic particle . . . . . . . . . . . . . . . . . . . . . . . . . . . 33
  3.3.2 The 「は」 topic particle . . . . . . . . . . . . . . . . . . . . . . . . . . . 33



# Using the Embedding Model to embedd the chunks

In [5]:
import numpy as np
from pathlib import Path

# Load the three JSONL files
KB_DIR = Path("../kb")
jsonl_files = {
    "eng_jap": KB_DIR / "eng_jap_chunks.jsonl",
    "style_guide": KB_DIR / "style_guide_chunks.jsonl",
    "grammar": KB_DIR / "grammar_chunks.jsonl",
}

# Load all records
loaded_data = {}
for name, path in jsonl_files.items():
    records = []
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                records.append(json.loads(line))
    loaded_data[name] = records
    print(f"{name}: loaded {len(records):,} records from {path.name}")

total_records = sum(len(v) for v in loaded_data.values())
print(f"\nTotal records across all sources: {total_records:,}")

eng_jap: loaded 46,616 records from eng_jap_chunks.jsonl
style_guide: loaded 181 records from style_guide_chunks.jsonl
grammar: loaded 81 records from grammar_chunks.jsonl

Total records across all sources: 46,878


In [14]:

# === BATCH EMBEDDING SCRIPT FOR FULL DATASET ===
# To embed all 46k+ eng_jap chunks, save and run this as a standalone script on a powerful machine

batch_embed_script = '''#!/usr/bin/env python3
"""Batch embed eng_jap_chunks.jsonl with BGE-M3 (optimized for large-scale processing)."""

import json
import sys
from pathlib import Path
from sentence_transformers import SentenceTransformer
import torch

KB_DIR = Path("kb") if Path("kb").exists() else Path("../kb")
input_path = KB_DIR / "eng_jap_chunks.jsonl"
output_path = KB_DIR / "eng_jap_chunks_embedded.jsonl"

if not input_path.exists():
    print(f"Error: {input_path} not found")
    sys.exit(1)

# Load model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Loading BGE-M3 model...")
model = SentenceTransformer("BAAI/bge-m3", trust_remote_code=True)
model.to(device)

# Read and process
print(f"Loading {input_path.name}...")
records = []
with open(input_path, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print(f"Loaded {len(records):,} records. Embedding...")

texts = [r.get("chunk_text", "") for r in records]
batch_size = 256 if device == "cuda" else 16

embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=batch_size,
    device=device,
)

# Add embeddings and save
print(f"Adding embeddings and saving...")
for record, embedding in zip(records, embeddings):
    record["embedding"] = embedding.tolist()

with open(output_path, "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\\n")

print(f"✓ Saved {len(records):,} embedded records to {output_path}")
'''

# Save the script
script_path = Path("../embed_eng_jap_full.py")
with open(script_path, "w", encoding="utf-8") as f:
    f.write(batch_embed_script)

print(f"Saved batch embedding script to {script_path.resolve()}")
print("\nTo embed all 46k+ eng_jap records:")
print(f"  python {script_path}")
print("\nNote: This will take ~30-60 minutes on CPU or ~5-10 minutes on GPU.")

Saved batch embedding script to D:\GitHub\is469\embed_eng_jap_full.py

To embed all 46k+ eng_jap records:
  python ..\embed_eng_jap_full.py

Note: This will take ~30-60 minutes on CPU or ~5-10 minutes on GPU.


In [9]:
%pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

# Load BGE-M3 embedding model (dense embeddings, multilingual)
print("Loading BAAI/bge-m3 model...")
model = SentenceTransformer("BAAI/bge-m3", trust_remote_code=True)
print(f"Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")

# Verify you can embed
test_embed = model.encode("Hello world", convert_to_numpy=True)
print(f"Test embedding shape: {test_embed.shape}")

Note: you may need to restart the kernel to use updated packages.


d:\GitHub\is469\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading BAAI/bge-m3 model...


d:\GitHub\is469\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Yun Lin\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 20908.15it/s]
d:\GitHub\is469\.venv\Lib\site-packages\huggingface_hub\file_do

Model loaded. Embedding dimension: 1024
Test embedding shape: (1024,)


In [12]:
import torch

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}\n")

# Move model to device
model.to(device)

print("Embedding chunks with BGE-M3...\n")
print("Note: Embedding full eng_jap (46k+) takes a long time on CPU.")
print("      Processing grammar + style_guide complete, plus 1k sample of eng_jap.\n")

embedded_data = {}

for source_name, records in loaded_data.items():
    if not records:
        print(f"  {source_name}: skipped (no records)")
        continue
    
    # For eng_jap, use only first 1000 records for demo
    if source_name == "eng_jap":
        records = records[:1000]
    
    print(f"  {source_name} ({len(records):,} records)...", flush=True)
    
    # Extract text to embed
    text_field = "chunk_text" if source_name == "eng_jap" else "text"
    texts = [r.get(text_field, "") for r in records]
    
    # Smaller batch size for CPU
    batch_size = 16
    
    try:
        # Embed all texts with progress
        embeddings = model.encode(
            texts,
            convert_to_numpy=True,
            show_progress_bar=True,
            batch_size=batch_size,
            device=device,
        )
        
        # Add embeddings to records
        for record, embedding in zip(records, embeddings):
            record["embedding"] = embedding.tolist()
        
        embedded_data[source_name] = records
        print(f"    ✓ Embedded shape: {embeddings.shape}\n")
    except Exception as e:
        print(f"    ✗ ERROR: {e}\n")
        embedded_data[source_name] = []

print("Demo embedding complete!")

Using device: cpu

Embedding chunks with BGE-M3...

Note: Embedding full eng_jap (46k+) takes a long time on CPU.
      Processing grammar + style_guide complete, plus 1k sample of eng_jap.

  eng_jap (1,000 records)...


Batches: 100%|██████████| 63/63 [08:56<00:00,  8.52s/it]

    ✓ Embedded shape: (1000, 1024)

  style_guide (181 records)...



Batches: 100%|██████████| 12/12 [00:52<00:00,  4.39s/it]

    ✓ Embedded shape: (181, 1024)

  grammar (81 records)...



Batches: 100%|██████████| 6/6 [00:42<00:00,  7.14s/it]

    ✓ Embedded shape: (81, 1024)

Demo embedding complete!


In [13]:
print("Saving embedded chunks to JSONL files...\n")

output_files = {
    "eng_jap": KB_DIR / "eng_jap_chunks_embedded.jsonl",
    "style_guide": KB_DIR / "style_guide_chunks_embedded.jsonl",
    "grammar": KB_DIR / "grammar_chunks_embedded.jsonl",
}

for source_name, out_path in output_files.items():
    records = embedded_data.get(source_name, [])
    if not records:
        print(f"  {source_name}: skipped (no records)")
        continue
    
    with open(out_path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    
    print(f"  ✓ {source_name:15s} → {len(records):6,} records to {out_path.name}")

print(f"\nAll embeddings saved to {KB_DIR}")

Saving embedded chunks to JSONL files...

  ✓ eng_jap         →  1,000 records to eng_jap_chunks_embedded.jsonl
  ✓ style_guide     →    181 records to style_guide_chunks_embedded.jsonl
  ✓ grammar         →     81 records to grammar_chunks_embedded.jsonl

All embeddings saved to ..\kb
